# Covariance & Correlation — companion notebook

Runnable companion for [`covariance-correlation.md`](./covariance-correlation.md).

1. Visualize joint-Gaussian density and 1$\sigma$ ellipses at $\rho \in \{-0.8, 0, 0.5, 0.95\}$.
2. Convergence of the sample covariance estimator as $n$ grows.
3. The classic $X \sim \mathcal{N}(0,1), Y = X^2$ counterexample: $\text{Cov}(X, Y) = 0$ but $Y$ is a deterministic function of $X$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)

## 1. Joint-Gaussian 1$\sigma$ ellipses

For $(X, Y) \sim \mathcal{N}(\mathbf{0}, \Sigma)$ with unit variances, the $1\sigma$ level set of the density is an ellipse whose axes are the eigenvectors of $\Sigma$ and lengths are $\sqrt{\lambda_i}$.

In [ ]:
def ellipse_points(cov, n=200):
    """1-sigma ellipse for a 2x2 covariance."""
    theta = np.linspace(0, 2*np.pi, n)
    circle = np.stack([np.cos(theta), np.sin(theta)])
    L = np.linalg.cholesky(cov)
    return L @ circle

rhos = [-0.8, 0.0, 0.5, 0.95]
fig, axes = plt.subplots(1, len(rhos), figsize=(4*len(rhos), 4), sharex=True, sharey=True)
for ax, rho in zip(axes, rhos):
    cov = np.array([[1.0, rho], [rho, 1.0]])
    pts = rng.multivariate_normal([0, 0], cov, size=2000)
    ax.scatter(pts[:, 0], pts[:, 1], s=4, alpha=0.3, color='tab:blue')
    ell = ellipse_points(cov)
    ax.plot(ell[0], ell[1], color='tab:red', lw=2, label='1σ ellipse')
    ax.set_title(f'ρ = {rho:+.2f}')
    ax.set_aspect('equal'); ax.set_xlim(-4, 4); ax.set_ylim(-4, 4)
    ax.axhline(0, color='gray', lw=0.5); ax.axvline(0, color='gray', lw=0.5)
    ax.legend(loc='lower right', fontsize=8)
plt.suptitle('Joint Gaussians at different correlations')
plt.tight_layout()
plt.show()

## 2. Sample covariance converges to the truth

Estimator error decays roughly as $1/\sqrt{n}$.

In [ ]:
true_cov = np.array([[1.0, 0.6, -0.2],
                     [0.6, 2.0,  0.1],
                     [-0.2, 0.1, 0.5]])
ns = np.unique(np.logspace(1.3, 4, 25).astype(int))
n_trials = 100
errs = np.zeros((len(ns), n_trials))
for i, n in enumerate(ns):
    for t in range(n_trials):
        s = rng.multivariate_normal(np.zeros(3), true_cov, size=n)
        hat = np.cov(s, rowvar=False, ddof=1)
        errs[i, t] = np.linalg.norm(hat - true_cov, 'fro')

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(ns, errs.mean(axis=1), marker='o', ms=4, label='mean Frobenius error')
ax.loglog(ns, 5/np.sqrt(ns), ls='--', color='gray', label=r'$\propto 1/\sqrt{n}$ reference')
ax.set_xlabel('sample size n'); ax.set_ylabel(r'$\|\hat\Sigma - \Sigma\|_F$')
ax.set_title('Sample covariance error decays as 1/√n')
ax.legend(); ax.grid(True, which='both', alpha=0.3)
plt.show()

## 3. Uncorrelated $\neq$ independent

Let $X \sim \mathcal{N}(0, 1)$ and $Y = X^2$. By symmetry $\text{Cov}(X, Y) = \mathbb{E}[X^3] - 0 = 0$, yet $Y$ is a deterministic function of $X$.

In [ ]:
X = rng.standard_normal(100_000)
Y = X**2

print(f'sample Pearson rho(X, Y)  = {np.corrcoef(X, Y)[0, 1]:+.4f}   (true 0)')
print(f'sample Cov(X, Y)          = {np.cov(X, Y, ddof=1)[0, 1]:+.4f}   (true 0)')
print(f'mutual information is clearly large: Y is a function of X')